# Diagnostics Case Gallery

This notebook uses small synthetic examples to show what `evaluation(..., collect_diagnostics=True)` emits under different mismatch scenarios.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "evaluation.py").exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

import evaluation as ev

pd.set_option("display.max_colwidth", 200)

## Synthetic Data Helpers

In [ ]:
def act(verb_idx, obj_idxs=None, act_type=1, related_acts=None):
    return {
        "act_idx": verb_idx,
        "obj_idxs": [obj_idxs or [], []],
        "act_type": act_type,
        "related_acts": related_acts or [],
    }


def make_sample(words, acts, pred, doc_id, text=None):
    text = text or " ".join(words) + "."
    return {
        "words": words,
        "acts": acts,
        "pred": pred,
        "sents": [words],
        "doc_id": doc_id,
        "docId": f"case:{doc_id}",
        "original_text": text,
    }


def evaluate_case(name, sample):
    metrics, diagnostics = ev.evaluation([sample], names=("case", "demo", name), collect_diagnostics=True)
    rows = []
    for row in diagnostics:
        rows.append({"case": name, **row})
    return {"case": name, "metrics": metrics, "diagnostics": rows}

## Define Cases

In [ ]:
cases = {
    "exact_match": make_sample(
        ["open", "file"],
        [act(0, [1])],
        [{"verb": "open", "arguments": ["file"]}],
        1,
        "open file.",
    ),
    "llm_missing_argument": make_sample(
        ["open", "file"],
        [act(0, [1])],
        [{"verb": "open", "arguments": []}],
        2,
        "open file.",
    ),
    "dataset_missing_argument": make_sample(
        ["open", "file"],
        [act(0, [])],
        [{"verb": "open", "arguments": ["file"]}],
        3,
        "open file.",
    ),
    "wrong_argument": make_sample(
        ["open", "file"],
        [act(0, [1])],
        [{"verb": "open", "arguments": ["folder"]}],
        4,
        "open file.",
    ),
    "dataset_split_head_words": make_sample(
        ["choose", "square", "shadow", "box"],
        [act(0, [1, 2, 3])],
        [{"verb": "choose", "arguments": ["square shadow box"]}],
        5,
        "choose square shadow box.",
    ),
    "unused_prediction_possible_dataset_missing_action": make_sample(
        ["open", "file", "save", "file"],
        [act(0, [1])],
        [{"verb": "open", "arguments": ["file"]}, {"verb": "save", "arguments": ["file"]}],
        6,
        "open file. save file.",
    ),
    "unused_prediction_llm_extra_only": make_sample(
        ["open", "file"],
        [act(0, [1])],
        [{"verb": "open", "arguments": ["file"]}, {"verb": "delete", "arguments": ["file"]}],
        7,
        "open file.",
    ),
    "unmatched_gold_action": make_sample(
        ["open", "file"],
        [act(0, [1])],
        [],
        8,
        "open file.",
    ),
    "wrong_action": make_sample(
        ["open", "file"],
        [act(0, [1])],
        [{"verb": "load", "arguments": ["file"]}],
        9,
        "open file.",
    ),
    "prediction_without_gold_action": make_sample(
        ["save", "file"],
        [],
        [{"verb": "save", "arguments": ["file"]}],
        10,
        "save file.",
    ),
}

## Run Diagnostics

In [ ]:
results = [evaluate_case(name, sample) for name, sample in cases.items()]

metrics_df = pd.DataFrame(
    [
        {
            "case": item["case"],
            "precision": item["metrics"][0],
            "recall": item["metrics"][1],
            "f1": item["metrics"][2],
            "obj_precision": item["metrics"][3],
            "obj_recall": item["metrics"][4],
            "obj_f1": item["metrics"][5],
            "num_diagnostics": len(item["diagnostics"]),
        }
        for item in results
    ]
)
diagnostics_df = pd.DataFrame([row for item in results for row in item["diagnostics"]])

metrics_df

In [ ]:
display_cols = [
    "case",
    "mismatch_type",
    "candidate_dataset_issue",
    "candidate_llm_issue",
    "reason",
    "gold_verb",
    "gold_arguments",
    "pred_verb",
    "pred_arguments",
    "original_text",
]
diagnostics_df[display_cols]

## Compact Case-to-Diagnosis Map

In [ ]:
case_map = (
    diagnostics_df.groupby("case")
    .agg(
        mismatch_types=("mismatch_type", lambda x: " | ".join(x)),
        dataset_issues=("candidate_dataset_issue", lambda x: " | ".join(v for v in x if isinstance(v, str) and v)),
        llm_issues=("candidate_llm_issue", lambda x: " | ".join(v for v in x if isinstance(v, str) and v)),
    )
    .reset_index()
)
metrics_df.merge(case_map, on="case", how="left").fillna("")

## Direct Argument Classifier Examples

In [ ]:
argument_cases = [
    ("llm_missing_argument", ["file"], []),
    ("dataset_missing_argument", [], ["file"]),
    ("wrong_argument", ["file"], ["folder"]),
    ("dataset_preposition_argument", ["to folder"], []),
    ("llm_preposition_argument", [], ["to folder"]),
    ("llm_split_head_words", ["square shadow box"], ["square", "shadow", "box"]),
]

rows = []
for name, gold_args, pred_args in argument_cases:
    info = ev.classify_argument_mismatch(gold_args, pred_args)
    rows.append({"case": name, "gold_args": gold_args, "pred_args": pred_args, **info})

pd.DataFrame(rows)